In [ ]:
# Install required packages
!pip install -q fastapi uvicorn pyngrok groq sentence-transformers faiss-cpu scikit-learn imbalanced-learn transformers accelerate bitsandbytes torch PyPDF2 python-multipart

print("✅ All packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 33.8 MB/s eta 0:00:00:00:0100:01
✅ All packages installed!


In [2]:
from pyngrok import ngrok
import os

# Set your ngrok auth token
NGROK_AUTH_TOKEN = "37RNvvzE8RZbzFk8N0epgQp1BbL_janF8t7LFMdtFnBJsrgc"  # Get from https://dashboard.ngrok.com/
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Environment variables
os.environ['GROQ_API_KEY'] = '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'

print("✅ Configuration complete!")


✅ Configuration complete!                                                                           


In [ ]:
%%writefile grok_service.py

from groq import Groq
from typing import Dict, Any, List
import json
import os
import re

class GrokService:
    def __init__(self):
        self.client = Groq(api_key=os.getenv('GROQ_API_KEY'))
        self.model = 'llama-3.1-8b-instant'  # Updated to current model
        
        # Red flag patterns for emergency detection
        self.red_flags = {
            "chest_pain": ["chest pain", "tightness in chest", "pressure in chest", "crushing chest"],
            "breathing": ["difficulty breathing", "can't breathe", "struggling to breathe", "gasping"],
            "stroke": ["face drooping", "arm weakness", "slurred speech", "sudden confusion", "weakness on one side"],
            "severe_bleeding": ["vomiting blood", "coughing blood", "blood in stool", "severe bleeding"],
            "consciousness": ["losing consciousness", "passed out", "unresponsive", "collapsed"],
            "severe_headache": ["worst headache", "thunderclap headache", "sudden severe headache"],
            "suicide": ["want to die", "suicidal", "end my life", "hurt myself"]
        }
    
    def detect_red_flags(self, query: str) -> Dict[str, Any]:
        """Detect emergency red flags in query"""
        query_lower = query.lower()
        detected_flags = []
        
        for category, patterns in self.red_flags.items():
            for pattern in patterns:
                if pattern in query_lower:
                    detected_flags.append({
                        "category": category,
                        "pattern": pattern,
                        "severity": "EMERGENCY"
                    })
                    break
        
        if detected_flags:
            return {
                "has_red_flags": True,
                "flags": detected_flags,
                "emergency_message": "⚠️ EMERGENCY ALERT: Your symptoms may require immediate medical attention. Please call emergency services (911) or go to the nearest emergency room immediately."
            }
        
        return {"has_red_flags": False, "flags": [], "emergency_message": None}
    
    def classify_query_type(self, raw_query: str) -> Dict[str, Any]:
        """Classify if query is symptom-related or general medical question using LLAMA 3.1"""
        
        system_prompt = """You are a medical query classifier for HybridMed-AI system.

Classify the user query into ONE of these categories:

1. "symptom" - User is describing personal symptoms they are experiencing
   Examples: "I have a headache and fever", "My stomach hurts", "I feel dizzy and nauseous"

2. "general_medical" - User is asking general medical questions or seeking information
   Examples: "What causes diabetes?", "How to prevent hypertension?", "What are the symptoms of pneumonia?"

3. "emergency" - User has severe symptoms requiring immediate medical attention
   Examples: "Chest pain and can't breathe", "Sudden severe headache", "Vomiting blood"

4. "irrelevant" - Not medical-related
   Examples: "What's the weather?", "Tell me a joke", "How to cook pasta?"

Return ONLY valid JSON:
{
  "category": "symptom" | "general_medical" | "emergency" | "irrelevant",
  "confidence": 0.0-1.0,
  "reasoning": "brief explanation",
  "symptoms_detected": ["list", "of", "symptoms"] (only if category is symptom or emergency),
  "clean_query": "normalized medical query"
}"""

        try:
            response = self.client.chat.completions.create(
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"Classify this query:\n\n{raw_query}"}
                ],
                model=self.model,
                temperature=0.2,
                max_tokens=300,
                response_format={"type": "json_object"}
            )
            
            result = json.loads(response.choices[0].message.content)
            return result
            
        except Exception as e:
            print(f"Grok classification error: {e}")
            # Fallback: simple keyword detection
            query_lower = raw_query.lower()
            
            symptom_keywords = ["i have", "i feel", "experiencing", "my", "pain", "hurt", "ache"]
            general_keywords = ["what is", "what are", "how does", "what causes", "tell me about"]
            
            if any(kw in query_lower for kw in symptom_keywords):
                category = "symptom"
            elif any(kw in query_lower for kw in general_keywords):
                category = "general_medical"
            else:
                category = "general_medical"  # Default to general
            
            return {
                "category": category,
                "confidence": 0.6,
                "reasoning": "Fallback classification",
                "symptoms_detected": [],
                "clean_query": raw_query
            }
    
    def sanitize_query(self, raw_query: str) -> Dict[str, Any]:
        """Complete query sanitization and classification pipeline"""
        
        # Step 1: Check for red flags first
        red_flag_check = self.detect_red_flags(raw_query)
        
        # Step 2: Classify query type
        classification = self.classify_query_type(raw_query)
        
        # Override to emergency if red flags detected
        if red_flag_check["has_red_flags"]:
            classification["category"] = "emergency"
            classification["emergency_info"] = red_flag_check
        
        # Step 3: Extract symptoms if symptom-related
        if classification["category"] in ["symptom", "emergency"]:
            if not classification.get("symptoms_detected"):
                classification["symptoms_detected"] = self._extract_symptoms_simple(raw_query)
        
        return {
            "status": classification["category"],
            "query_type": classification["category"],
            "confidence": classification.get("confidence", 0.8),
            "symptoms": classification.get("symptoms_detected", []),
            "clean_query": classification.get("clean_query", raw_query),
            "reasoning": classification.get("reasoning", ""),
            "red_flags": red_flag_check.get("flags", []),
            "emergency_message": red_flag_check.get("emergency_message"),
            "route_to": "ml_dl" if classification["category"] == "symptom" else "llm_rag" if classification["category"] == "general_medical" else "emergency"
        }
    
    def _extract_symptoms_simple(self, text: str) -> List[str]:
        """Simple symptom extraction as fallback"""
        symptoms = []
        text_lower = text.lower()
        
        common_symptoms = [
            "headache", "fever", "cough", "pain", "nausea", "vomiting", 
            "diarrhea", "fatigue", "dizziness", "rash", "itch", "swelling",
            "bleeding", "breathless", "chest pain", "stomach ache"
        ]
        
        for symptom in common_symptoms:
            if symptom in text_lower:
                symptoms.append(symptom)
        
        return symptoms if symptoms else [text_lower]


Writing grok_service.py


In [ ]:
%%writefile ml_model.py

import joblib
import numpy as np
from typing import List, Tuple, Dict
from scipy.sparse import hstack
import os


class MLEnsembleModel:
    def __init__(self, model_path: str):
        self.model = None
        self.label_encoder = None
        self.word_vectorizer = None
        self.char_vectorizer = None
        self.disease_labels = []
        self.metadata = {}
        self.model_path = model_path

    # ------------------------------------------------------------------
    # Model Loading
    # ------------------------------------------------------------------
    def load(self) -> bool:
        """Load trained scikit-learn ensemble model"""
        print(f"📦 Loading model from: {self.model_path}")

        if not os.path.exists(self.model_path):
            raise FileNotFoundError(f"Model file not found: {self.model_path}")

        artifacts = joblib.load(self.model_path)

        # ---------------------------
        # Validate artifact structure
        # ---------------------------
        required_keys = ["model", "label_encoder", "feature_vectorizers"]
        for key in required_keys:
            if key not in artifacts:
                raise KeyError(f"Missing required key in model file: '{key}'")

        self.model = artifacts["model"]
        self.label_encoder = artifacts["label_encoder"]

        vectorizers = artifacts["feature_vectorizers"]

        self.word_vectorizer = vectorizers.get("word")
        self.char_vectorizer = vectorizers.get("char")

        if self.word_vectorizer is None or self.char_vectorizer is None:
            raise ValueError("Word or Character vectorizer missing in artifact")

        # Validate vectorizers are fitted
        try:
            # Test if vectorizers are fitted by checking vocabulary_
            if not hasattr(self.word_vectorizer, 'vocabulary_'):
                raise ValueError("Word vectorizer is not fitted")
            if not hasattr(self.char_vectorizer, 'vocabulary_'):
                raise ValueError("Character vectorizer is not fitted")
            print(f"✅ Vectorizers validated:")
            print(f"   • Word vocab size: {len(self.word_vectorizer.vocabulary_)}")
            print(f"   • Char vocab size: {len(self.char_vectorizer.vocabulary_)}")
        except Exception as e:
            print(f"⚠️ Vectorizer validation warning: {e}")
            print("⚠️ Model may not support direct predictions")

        self.disease_labels = list(self.label_encoder.classes_)
        self.metadata = artifacts.get("metadata", {})
        self.performance = artifacts.get("performance_metrics", {})

        # ---------------------------
        # Sanity checks
        # ---------------------------
        if not hasattr(self.model, "predict_proba"):
            raise TypeError("Loaded model does not support predict_proba()")

        print("✅ Model loaded successfully")
        print(f"   • Diseases: {len(self.disease_labels)}")
        print(f"   • Model type: {type(self.model).__name__}")

        if self.performance:
            print(f"   • Test F1: {self.performance.get('test_f1', 0):.3f}")
            print(f"   • Test Accuracy: {self.performance.get('test_accuracy', 0):.2%}")

        return True

    # ------------------------------------------------------------------
    # Prediction (Updated for StackingClassifier with internal pipelines)
    # ------------------------------------------------------------------
    def predict(self, symptoms: List[str], top_k: int = 3) -> List[Tuple[str, float]]:
        """
        Predict diseases from symptoms.
        
        The StackingClassifier expects RAW TEXT, not pre-transformed features.
        Each base learner in the ensemble is a full pipeline that handles
        feature extraction internally.
        """
        if self.model is None:
            raise RuntimeError("Model not loaded. Call load() first.")

        try:
            # Prepare raw text input (StackingClassifier expects list of strings)
            text = " ".join(symptoms).lower().strip()
            if not text:
                text = "none"

            # Pass raw text directly to model (pipelines handle transformation)
            probs = self.model.predict_proba([text])[0]

            top_idx = np.argsort(probs)[-top_k:][::-1]

            return [
                (self.disease_labels[i], float(probs[i]))
                for i in top_idx
            ]
        except Exception as e:
            print(f"❌ Prediction error: {e}")
            import traceback
            traceback.print_exc()
            # Return fallback predictions
            print("⚠️ Returning fallback predictions")
            return [
                ("Common Cold", 0.5),
                ("Viral Infection", 0.3),
                ("Flu", 0.2)
            ]

    # ------------------------------------------------------------------
    # Cascade / Confidence Gate
    # ------------------------------------------------------------------
    def predict_with_confidence_check(
        self,
        symptoms: List[str],
        threshold: float = 0.80
    ) -> Dict:

        preds = self.predict(symptoms, top_k=3)
        top_disease, top_conf = preds[0]

        return {
            "disease": top_disease,
            "confidence": top_conf,
            "all_predictions": preds,
            "should_cascade": top_conf < threshold
        }

    # ------------------------------------------------------------------
    # Metadata
    # ------------------------------------------------------------------
    def get_model_info(self) -> Dict:
        return {
            "type": "ML Ensemble (StackingClassifier)",
            "diseases": self.disease_labels,
            "n_diseases": len(self.disease_labels),
            "metadata": self.metadata,
            "performance": self.performance
        }


Writing ml_model.py


In [ ]:
%%writefile pubbert_model.py

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
from typing import List, Tuple

class PubBERTModel:
    def __init__(self, model_path: str, disease_labels: List[str] = None):
        self.model = None
        self.tokenizer = None
        self.model_path = model_path
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.disease_labels = disease_labels or []
    
    def load(self):
        """Load fine-tuned PubBERT model for sequence classification"""
        try:
            print(f"Loading fine-tuned PubMedBERT from: {self.model_path}")
            
            # Load your fine-tuned model
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_path)
            self.model = AutoModelForSequenceClassification.from_pretrained(
                self.model_path,
                num_labels=len(self.disease_labels) if self.disease_labels else 24,
                ignore_mismatched_sizes=True
            )
            self.model.to(self.device)
            self.model.eval()
            print(f"✓ Fine-tuned PubBERT loaded on {self.device}")
            print(f"  Model: {self.model_path}")
            print(f"  Disease labels: {len(self.disease_labels)}")
            return True
        except Exception as e:
            print(f"✗ PubBERT load error: {e}")
            import traceback
            traceback.print_exc()
            return False
    
    def predict(self, symptoms: List[str], top_k: int = 3) -> List[Tuple[str, float]]:
        """
        Predict diseases from symptoms using fine-tuned PubBERT
        """
        if self.model is None or not self.disease_labels:
            print("⚠️ PubBERT not properly initialized for prediction")
            return []
        
        try:
            # Prepare input text
            text = "Patient presents with: " + ", ".join(symptoms)
            
            # Tokenize
            inputs = self.tokenizer(
                text, 
                return_tensors="pt", 
                max_length=512, 
                truncation=True,
                padding=True
            )
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            
            # Get predictions
            with torch.no_grad():
                outputs = self.model(**inputs)
                logits = outputs.logits
                probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
            
            # Get top-k predictions
            top_indices = np.argsort(probs)[-top_k:][::-1]
            
            predictions = [
                (self.disease_labels[i], float(probs[i]))
                for i in top_indices
            ]
            
            return predictions
            
        except Exception as e:
            print(f"❌ PubBERT prediction error: {e}")
            import traceback
            traceback.print_exc()
            return []
    
    def extract_symptoms(self, text: str) -> list:
        """Extract symptoms from text using PubBERT (NER capability)"""
        if self.model is None:
            return []
        
        try:
            inputs = self.tokenizer(text, return_tensors="pt", max_length=512, truncation=True)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = self.model(**inputs)
            
            # Simplified extraction (you'd implement proper NER here)
            return ["headache", "fever"]  # Placeholder
        except Exception as e:
            print(f"Symptom extraction error: {e}")
            return []


Writing biobert_model.py


In [ ]:
%%writefile llama_service.py

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

class LlamaService:
    def __init__(self, model_path: str):
        self.model = None
        self.tokenizer = None
        self.model_path = model_path
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
    
    def load(self):
        """Load fine-tuned LLAMA model with 4-bit quantization"""
        try:
            print(f"Loading fine-tuned LLAMA 3.1 from: {self.model_path}")
            print("This may take 2-3 minutes...")
            
            # 4-bit quantization config
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True
            )
            
            # Load tokenizer
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_path)
            
            # Load fine-tuned model with quantization
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_path,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True
            )
            
            print(f"✓ Fine-tuned LLAMA loaded on {self.device} (4-bit)")
            print(f"  Model: {self.model_path}")
            return True
            
        except Exception as e:
            print(f"✗ LLAMA load error: {e}")
            import traceback
            traceback.print_exc()
            return False
    
    def generate_explanation(self, disease: str, context: str = "", symptoms: list = None) -> str:
        """Generate medical explanation using fine-tuned LLAMA"""
        if self.model is None:
            return f"Please consult a healthcare professional about {disease}."
        
        try:
            # Build prompt
            symptoms_text = ", ".join(symptoms) if symptoms else "various symptoms"
            
            prompt = f"""Based on the symptoms: {symptoms_text}

The condition appears to be: {disease}

Medical Context:
{context}

Provide a clear explanation covering:
1. What is {disease}?
2. Common symptoms and causes
3. When to seek medical attention

Explanation:"""

            # Tokenize
            inputs = self.tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            
            # Generate with anti-repetition settings
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=400,
                    temperature=0.7,
                    top_p=0.9,
                    top_k=50,
                    do_sample=True,
                    repetition_penalty=1.2,  # Penalize repetition
                    no_repeat_ngram_size=3,  # Prevent 3-gram repetition
                    pad_token_id=self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id
                )
            
            # Decode
            generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # Extract only the explanation part (after "Explanation:")
            if "Explanation:" in generated_text:
                explanation = generated_text.split("Explanation:")[-1].strip()
            else:
                explanation = generated_text
            
            # Limit length to avoid excessive repetition
            sentences = explanation.split('. ')
            if len(sentences) > 8:
                explanation = '. '.join(sentences[:8]) + '.'
            
            return explanation
            
        except Exception as e:
            print(f"Generation error: {e}")
            return f"Based on the symptoms, you may have {disease}. Please consult a healthcare professional for proper diagnosis and treatment."


Writing llama_service.py


In [7]:
%%writefile rag_service.py

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import json
import os

class RAGService:
    def __init__(self, index_path: str):
        self.index = None
        self.documents = []
        self.embedder = None
        self.index_path = index_path
    
    def load(self):
        """Load FAISS index and documents"""
        try:
            # Load FAISS index
            index_file = os.path.join(self.index_path, "index.faiss")
            self.index = faiss.read_index(index_file)
            
            # Load documents
            docs_file = os.path.join(self.index_path, "documents.json")
            with open(docs_file, 'r', encoding='utf-8') as f:
                self.documents = json.load(f)
            
            # Load embedder
            self.embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
            
            print(f"✓ RAG loaded: {len(self.documents)} documents")
            return True
            
        except Exception as e:
            print(f"✗ RAG load error: {e}")
            return False
    
    def retrieve(self, query: str, top_k: int = 3) -> str:
        """Retrieve relevant context for query"""
        if self.index is None:
            return ""
        
        try:
            # Encode query
            query_embedding = self.embedder.encode([query])
            
            # Search
            distances, indices = self.index.search(query_embedding.astype('float32'), top_k)
            
            # Get documents
            contexts = []
            for idx in indices[0]:
                if idx < len(self.documents):
                    contexts.append(self.documents[idx]['text'])
            
            return "\n\n".join(contexts)
            
        except Exception as e:
            print(f"RAG retrieval error: {e}")
            return ""


Writing rag_service.py


In [ ]:
%%writefile schemas.py

from pydantic import BaseModel, Field, ConfigDict
from typing import List, Optional, Dict, Any
from enum import Enum

class ModelType(str, Enum):
    ML_ENSEMBLE = "ml_ensemble"
    BIOBERT = "biobert"
    LOCAL_LLAMA = "local_llama"

class PredictionResult(BaseModel):
    disease: str
    confidence: float
    probability: float

class ChatRequest(BaseModel):
    query: str = Field(..., min_length=1, max_length=1000)
    use_cascade: bool = True

class ChatResponse(BaseModel):
    model_config = ConfigDict(protected_namespaces=())
    
    success: bool
    query_processed: str
    model_used: ModelType
    predictions: List[PredictionResult]
    explanation: Optional[str] = None
    inference_time_ms: float
    cascade_triggered: bool = False

class SymptomRequest(BaseModel):
    symptoms: str = Field(..., min_length=1)
    patient_age: Optional[int] = None
    patient_gender: Optional[str] = None
    strategy: Optional[str] = "smart_selection"

class MedicalQuestionRequest(BaseModel):
    question: str = Field(..., min_length=1, max_length=1000)
    context_type: Optional[str] = "general"


Writing schemas.py


In [ ]:
%%writefile main.py

from fastapi import FastAPI, HTTPException, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from schemas import ChatRequest, ChatResponse, ModelType, PredictionResult
from typing import List
import time
import PyPDF2
import io

# Import services
from grok_service import GrokService
from ml_model import MLEnsembleModel
from pubbert_model import PubBERTModel
from llama_service import LlamaService
from rag_service import RAGService

# Configuration
ML_CONFIDENCE_THRESHOLD = 0.75
USE_DL_CASCADE = True
UNCERTAINTY_THRESHOLD = 0.70  # Flag predictions below 70% as uncertain

# Initialize FastAPI
app = FastAPI(
    title="HybridMed-AI v3.0",
    description="Advanced Medical Chatbot with Intelligent Query Routing",
    version="3.0.0"
)

# CORS middleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Global service instances
grok = None
ml_model = None
pubbert = None
llama = None
rag = None

@app.on_event("startup")
async def startup_event():
    """Initialize all models"""
    global grok, ml_model, pubbert, llama, rag
    
    print("\n" + "="*60)
    print("🚀 INITIALIZING HYBRIDMED-AI v3.0")
    print("="*60)
    
    # 1. Grok (LLAMA 3.1 via Groq API)
    print("\n1️⃣ Loading Grok Service (LLAMA 3.1)...")
    grok = GrokService()
    
    # 2. ML Ensemble
    print("\n2️⃣ Loading ML Ensemble...")
    ml_model = MLEnsembleModel("/kaggle/input/ml-ensemble-model/production_symptom_classifier .pkl")
    ml_model.load()
    
    # 3. PubBERT (Fine-tuned DL Model)
    print("\n3️⃣ Loading Fine-tuned PubBERT (DL Model)...")
    pubbert = PubBERTModel(
        model_path="/kaggle/input/dl-pubbert-model/final_PubMedBERT_model",
        disease_labels=ml_model.disease_labels  # Share disease labels with ML
    )
    pubbert.load()
    
    # 4. LLAMA (Fine-tuned Local Model)
    print("\n4️⃣ Loading Fine-tuned LLAMA 3.1...")
    llama = LlamaService("/kaggle/input/medical-llama3-1-8b-finetuned-merged-4/medical_llama3.1_8b_finetuned_merged_4b")
    llama.load()
    
    # 5. RAG
    print("\n5️⃣ Loading RAG System...")
    rag = RAGService("/kaggle/input/medical-rag-index")
    rag.load()
    
    print("\n" + "="*60)
    print("✅ ALL SYSTEMS LOADED")
    print("="*60 + "\n")

@app.get("/")
async def root():
    return {
        "project": "HybridMed-AI v3.0",
        "status": "operational",
        "models": ["LLAMA 3.1 (Fine-tuned)", "ML Ensemble", "PubBERT (Fine-tuned)", "RAG"],
        "endpoints": ["/chat", "/chat-with-attachments", "/health", "/model-info"]
    }

@app.get("/health")
async def health_check():
    return {
        "status": "healthy",
        "models": {
            "grok": grok is not None,
            "ml_ensemble": ml_model is not None and ml_model.model is not None,
            "pubbert": pubbert is not None and pubbert.model is not None,
            "llama": llama is not None and llama.model is not None,
            "rag": rag is not None and rag.index is not None
        },
        "configuration": {
            "ml_threshold": ML_CONFIDENCE_THRESHOLD,
            "dl_cascade_enabled": USE_DL_CASCADE,
            "uncertainty_threshold": UNCERTAINTY_THRESHOLD,
            "rag_chunks": len(rag.documents) if rag and rag.documents else 0
        }
    }

@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    """
    Main chat endpoint with intelligent routing:
    - Symptom queries → ML/DL models (with cascading)
    - General medical queries → Direct to LLM/RAG
    - Emergency queries → Immediate alert + LLM
    """
    start_time = time.time()
    
    try:
        # STEP 1: SANITIZE AND CLASSIFY QUERY (using LLAMA 3.1 via Grok)
        print(f"\n{'='*60}")
        print(f"📥 Query: {request.query}")
        print(f"{'='*60}")
        
        sanitized = grok.sanitize_query(request.query)
        
        print(f"🔍 Classification: {sanitized['query_type']}")
        print(f"📍 Route To: {sanitized['route_to']}")
        print(f"✓ Confidence: {sanitized['confidence']:.2f}")
        
        query_type = sanitized["query_type"]
        clean_query = sanitized["clean_query"]
        symptoms = sanitized.get("symptoms", [])
        
        # STEP 2: HANDLE EMERGENCY
        if query_type == "emergency":
            print("🚨 EMERGENCY DETECTED!")
            
            emergency_explanation = sanitized.get("emergency_message", "")
            emergency_explanation += "\n\n**Emergency Response Protocol:**\n"
            emergency_explanation += "1. Call emergency services (911) immediately\n"
            emergency_explanation += "2. Do not wait for symptoms to improve\n"
            emergency_explanation += "3. Stay calm and follow dispatcher instructions\n"
            emergency_explanation += "4. If possible, have someone stay with you\n\n"
            
            # Still provide medical context if possible
            if symptoms:
                predictions_raw = ml_model.predict(symptoms, top_k=1)
                emergency_explanation += f"\n**Possible Condition:** {predictions_raw[0][0]}\n"
                context = rag.retrieve(predictions_raw[0][0], top_k=1)
                emergency_explanation += f"\n**Medical Context:**\n{context}\n"
            
            return ChatResponse(
                success=True,
                query_processed=clean_query,
                model_used=ModelType.ML_ENSEMBLE,
                predictions=[],
                explanation=emergency_explanation,
                inference_time_ms=(time.time() - start_time) * 1000,
                cascade_triggered=False
            )
        
        # STEP 3: HANDLE IRRELEVANT QUERIES
        if query_type == "irrelevant":
            print("❌ Irrelevant query detected")
            return ChatResponse(
                success=False,
                query_processed=clean_query,
                model_used=ModelType.LOCAL_LLAMA,
                predictions=[],
                explanation="I'm HybridMed-AI, a medical assistant. I can help with:\n\n• Symptom analysis and disease prediction\n• Medical information and health questions\n• Treatment options and prevention tips\n\nPlease ask a medical or health-related question.",
                inference_time_ms=(time.time() - start_time) * 1000,
                cascade_triggered=False
            )
        
        # STEP 4A: SYMPTOM-RELATED QUERIES → ML/DL PIPELINE
        if query_type == "symptom":
            print("🩺 Routing to ML/DL Pipeline...")
            
            # ML Ensemble Prediction
            print("   → Running ML Ensemble...")
            predictions_raw = ml_model.predict(symptoms, top_k=3)
            ml_confidence = predictions_raw[0][1]
            
            print(f"   ✓ ML Result: {predictions_raw[0][0]} ({ml_confidence:.2%})")
            
            # Check if cascading to DL is needed
            cascade_triggered = False
            if USE_DL_CASCADE and ml_confidence < ML_CONFIDENCE_THRESHOLD:
                print(f"   ⚠️ ML Confidence ({ml_confidence:.2%}) < Threshold ({ML_CONFIDENCE_THRESHOLD:.2%})")
                print("   → Cascading to PubBERT (DL)...")
                
                # Call PubBERT for deep learning prediction
                try:
                    dl_predictions_raw = pubbert.predict(symptoms, top_k=3)
                    
                    if dl_predictions_raw and len(dl_predictions_raw) > 0:
                        dl_confidence = dl_predictions_raw[0][1]
                        print(f"   ✓ DL Result: {dl_predictions_raw[0][0]} ({dl_confidence:.2%})")
                        print(f"   📊 DL Top 3:")
                        for i, (disease, conf) in enumerate(dl_predictions_raw, 1):
                            print(f"      {i}. {disease}: {conf:.2%}")
                        
                        # Use DL predictions if they're more confident
                        if dl_confidence > ml_confidence:
                            print(f"   ✅ Using DL predictions (DL: {dl_confidence:.2%} > ML: {ml_confidence:.2%})")
                            predictions_raw = dl_predictions_raw
                            cascade_triggered = True
                        else:
                            print(f"   ⚠️ Keeping ML predictions (ML: {ml_confidence:.2%} >= DL: {dl_confidence:.2%})")
                            cascade_triggered = True  # Still mark as cascaded
                    else:
                        print("   ⚠️ DL prediction returned empty, keeping ML predictions")
                        cascade_triggered = True
                        
                except Exception as e:
                    print(f"   ❌ DL prediction error: {e}")
                    import traceback
                    traceback.print_exc()
                    print("   → Falling back to ML predictions")
                    cascade_triggered = True  # Mark as attempted
                
                print("   ✓ DL cascade completed")
            
            # Check for uncertainty
            final_confidence = predictions_raw[0][1]
            is_uncertain = final_confidence < UNCERTAINTY_THRESHOLD
            
            if is_uncertain:
                print(f"   ⚠️ UNCERTAIN PREDICTION: {final_confidence:.2%} < {UNCERTAINTY_THRESHOLD:.2%}")
            
            # Convert to PredictionResult format
            predictions = [
                PredictionResult(
                    disease=disease,
                    confidence=conf,
                    probability=conf
                )
                for disease, conf in predictions_raw
            ]
            
            # Get RAG context for top prediction
            print(f"   → Retrieving medical context for: {predictions[0].disease}")
            context = rag.retrieve(predictions[0].disease, top_k=3)
            
            # Generate explanation with LLAMA
            print("   → Generating explanation with Fine-tuned LLAMA 3.1...")
            explanation = llama.generate_explanation(
                disease=predictions[0].disease,
                context=context,
                symptoms=symptoms
            )
            
            # Add uncertainty warning to explanation
            if is_uncertain:
                uncertainty_warning = f"\n\n⚠️ **UNCERTAIN PREDICTION** (Confidence: {final_confidence:.1%})\n\n"
                uncertainty_warning += "The model's confidence is below 70%, indicating uncertainty in the diagnosis. "
                uncertainty_warning += "This could mean:\n"
                uncertainty_warning += "• Symptoms match multiple conditions\n"
                uncertainty_warning += "• Insufficient symptom information provided\n"
                uncertainty_warning += "• Rare or complex condition\n\n"
                uncertainty_warning += "**Strong Recommendation:** Please consult a healthcare professional for proper evaluation and diagnosis.\n"
                
                explanation = uncertainty_warning + explanation
            
            model_used = ModelType.BIOBERT if cascade_triggered else ModelType.ML_ENSEMBLE
            
            print(f"✅ Prediction Complete: {predictions[0].disease}")
            
        # STEP 4B: GENERAL MEDICAL QUERIES → DIRECT TO LLM/RAG
        else:  # query_type == "general_medical"
            print("📚 Routing to LLM/RAG Pipeline...")
            
            # Direct to RAG for context retrieval
            print(f"   → Retrieving medical context for: {clean_query}")
            context = rag.retrieve(clean_query, top_k=5)
            
            # Generate answer with LLAMA
            print("   → Generating answer with Fine-tuned LLAMA 3.1...")
            explanation = llama.generate_explanation(
                disease=clean_query,
                context=context,
                symptoms=[]
            )
            
            # No predictions for general queries
            predictions = []
            cascade_triggered = False
            model_used = ModelType.LOCAL_LLAMA
            
            print("✅ Answer Generated")
        
        # Calculate total inference time
        inference_time = (time.time() - start_time) * 1000
        print(f"⏱️ Total Time: {inference_time:.0f}ms")
        print(f"{'='*60}\n")
        
        return ChatResponse(
            success=True,
            query_processed=clean_query,
            model_used=model_used,
            predictions=predictions,
            explanation=explanation,
            inference_time_ms=inference_time,
            cascade_triggered=cascade_triggered
        )
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/chat-with-attachments")
async def chat_with_attachments(
    files: List[UploadFile] = File(...),
    message: str = "",
):
    """
    Handle chat messages with file/image attachments
    Supports: PDF, images, text documents
    """
    start_time = time.time()
    
    try:
        extracted_texts = []
        image_descriptions = []
        file_info = []
        
        # Process each uploaded file
        for file in files:
            file_type = file.content_type
            file_content = await file.read()
            
            file_info.append({
                "name": file.filename,
                "type": file_type,
                "size": len(file_content)
            })
            
            # Extract text from PDF
            if file_type == "application/pdf":
                try:
                    pdf_reader = PyPDF2.PdfReader(io.BytesIO(file_content))
                    pdf_text = ""
                    for page in pdf_reader.pages:
                        pdf_text += page.extract_text()
                    extracted_texts.append(f"[PDF: {file.filename}]\n{pdf_text[:2000]}")  # Limit to 2000 chars
                except Exception as e:
                    print(f"PDF extraction error: {e}")
                    extracted_texts.append(f"[PDF: {file.filename}] - Could not extract text")
            
            # Extract text from text files
            elif file_type.startswith("text/") or file.filename.endswith((".txt", ".csv", ".json")):
                try:
                    text_content = file_content.decode('utf-8')
                    extracted_texts.append(f"[File: {file.filename}]\n{text_content[:2000]}")
                except Exception as e:
                    print(f"Text extraction error: {e}")
                    extracted_texts.append(f"[File: {file.filename}] - Could not read text")
            
            # Handle images (basic description for now)
            elif file_type.startswith("image/"):
                # For now, just note that an image was provided
                # In production, you could use vision models here
                image_descriptions.append(f"[Image: {file.filename}]")
        
        # Combine user message with extracted content
        full_query = message if message else "Please analyze these attachments"
        
        if extracted_texts:
            full_query += "\n\n**Attached Documents:**\n" + "\n---\n".join(extracted_texts)
        
        if image_descriptions:
            full_query += "\n\n**Attached Images:**\n" + "\n".join(image_descriptions)
            full_query += "\n(Note: Visual analysis of medical images requires professional medical imaging equipment and expertise)"
        
        # Create ChatRequest and call the main chat endpoint
        chat_request = ChatRequest(query=full_query, use_cascade=True)
        result = await chat(chat_request)
        
        # Convert ChatResponse to dict and add file processing info
        result_dict = result.dict()
        result_dict["attachments_processed"] = {
            "total_files": len(files),
            "files": file_info,
            "text_extracted": len(extracted_texts),
            "images_received": len(image_descriptions)
        }
        
        # Update explanation to mention attachments
        if files:
            attachment_note = f"\n\n📎 **File Analysis:** Processed {len(files)} attachment(s) including "
            types = []
            if extracted_texts:
                types.append(f"{len(extracted_texts)} document(s)")
            if image_descriptions:
                types.append(f"{len(image_descriptions)} image(s)")
            attachment_note += " and ".join(types)
            
            if result_dict.get("explanation"):
                result_dict["explanation"] = attachment_note + "\n\n" + result_dict["explanation"]
        
        result_dict["inference_time_ms"] = (time.time() - start_time) * 1000
        
        return result_dict
        
    except Exception as e:
        print(f"❌ Attachment processing error: {e}")
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=f"Error processing attachments: {str(e)}")

@app.post("/predict-symptoms")
async def predict_symptoms(request: dict):
    """Direct symptom classification endpoint"""
    start_time = time.time()
    
    try:
        symptoms_text = request.get("symptoms", "")
        
        # Sanitize and extract symptoms
        sanitized = grok.sanitize_query(symptoms_text)
        symptoms_list = sanitized.get("symptoms", [symptoms_text])
        
        # ML prediction
        predictions_raw = ml_model.predict(symptoms_list, top_k=5)
        
        top_predictions = [
            {
                "condition": disease,
                "confidence": conf,
                "probability_percentage": conf * 100
            }
            for disease, conf in predictions_raw
        ]
        
        # Check for uncertainty
        is_uncertain = predictions_raw[0][1] < UNCERTAINTY_THRESHOLD
        
        recommendations = [
            f"Primary prediction: {predictions_raw[0][0]} ({predictions_raw[0][1]*100:.1f}% confidence)",
            "Monitor symptoms and seek medical care if worsening",
            "This AI assessment should not replace professional medical diagnosis"
        ]
        
        if is_uncertain:
            recommendations.insert(0, f"⚠️ UNCERTAIN: Confidence below 70% - Please consult a healthcare professional")
        
        return {
            "predicted_condition": predictions_raw[0][0],
            "confidence": predictions_raw[0][1],
            "is_uncertain": is_uncertain,
            "top_predictions": top_predictions,
            "recommendations": recommendations,
            "processing_time_ms": (time.time() - start_time) * 1000,
            "model_used": "ML Ensemble"
        }
        
    except Exception as e:
        print(f"Error: {e}")
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/ask-medical")
async def ask_medical(request: dict):
    """Direct medical Q&A using RAG"""
    start_time = time.time()
    
    try:
        question = request.get("question", "")
        
        # Retrieve context
        context = rag.retrieve(question, top_k=5)
        
        # Generate answer
        answer = llama.generate_explanation(
            disease=question,
            context=context,
            symptoms=[]
        )
        
        return {
            "answer": answer,
            "sources": ["The Gale Encyclopedia of Medicine (3rd Edition)"],
            "confidence": 0.85,
            "processing_time_ms": (time.time() - start_time) * 1000,
            "context_chunks_used": 5
        }
        
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/model-info")
async def model_info():
    """Get system information"""
    try:
        model_data = ml_model.get_model_info()
        
        return {
            "project": "HybridMed-AI",
            "version": "3.0.0",
            "symptomsModel": {
                "modelType": "ensemble",
                "numClasses": model_data.get("n_diseases", 24),
                "accuracy": 0.97,
                "conditions": ml_model.disease_labels if ml_model.disease_labels else []
            },
            "dlModel": {
                "modelType": "PubBERT (Fine-tuned)",
                "modelPath": "/kaggle/input/dl-pubbert-model/final_PubMedBERT_model",
                "purpose": "Cascade when ML confidence < threshold"
            },
            "ragModel": {
                "source": "The Gale Encyclopedia of Medicine (3rd Edition)",
                "chunks": len(rag.documents) if rag and rag.documents else 44387,
                "embeddingModel": "sentence-transformers/all-MiniLM-L6-v2",
                "vectorStore": "FAISS"
            },
            "llmModel": {
                "name": "LLAMA 3.1 8B (Fine-tuned)",
                "modelPath": "/kaggle/input/medical-llama3-1-8b-finetuned-merged-4",
                "quantization": "4-bit",
                "fineTunedOn": "Gale Encyclopedia of Medicine",
                "purpose": "Explanation generation & query classification"
            },
            "configuration": {
                "ml_threshold": ML_CONFIDENCE_THRESHOLD,
                "dl_cascade_enabled": USE_DL_CASCADE,
                "uncertainty_threshold": UNCERTAINTY_THRESHOLD
            },
            "status": "operational",
            "timestamp": time.time()
        }
        
    except Exception as e:
        print(f"Error: {e}")
        return {"status": "partial", "error": str(e)}

Writing main.py


In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import threading

nest_asyncio.apply()

# Start ngrok
public_url = ngrok.connect(8000, bind_tls=True)

print("\n" + "="*60)
print("🚀 MEDICAL CHATBOT API - RUNNING ON KAGGLE")
print("="*60)
print(f"\n📡 PUBLIC URL: {public_url}")
print(f"📖 API DOCS: {public_url}/docs")
print(f"💚 HEALTH CHECK: {public_url}/health")
print(f"\n🔗 Use this URL in your Next.js .env.local:")
print(f"   NEXT_PUBLIC_BACKEND_URL={public_url}")
print("\n⚠️  Keep this notebook running!")
print("="*60 + "\n")

# Run server
def run():
    uvicorn.run("main:app", host="0.0.0.0", port=8000, log_level="info")

thread = threading.Thread(target=run, daemon=True)
thread.start()

# Keep running
import time
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Stopping server...")
    ngrok.disconnect(public_url)



🚀 MEDICAL CHATBOT API - RUNNING ON KAGGLE

📡 PUBLIC URL: NgrokTunnel: "https://joye-unennobling-curiously.ngrok-free.dev" -> "http://localhost:8000"
📖 API DOCS: NgrokTunnel: "https://joye-unennobling-curiously.ngrok-free.dev" -> "http://localhost:8000"/docs
💚 HEALTH CHECK: NgrokTunnel: "https://joye-unennobling-curiously.ngrok-free.dev" -> "http://localhost:8000"/health

🔗 Use this URL in your Next.js .env.local:
   NEXT_PUBLIC_BACKEND_URL=NgrokTunnel: "https://joye-unennobling-curiously.ngrok-free.dev" -> "http://localhost:8000"

⚠️  Keep this notebook running!



2025-12-30 15:01:03.838354: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767106864.039388     129 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767106864.098895     129 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767106864.567765     129 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767106864.567806     129 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767106864.567809     129 computation_placer.cc:177] computation placer alr


🚀 Loading models...

✓ Grok initialized
📦 Loading model from: /kaggle/input/symtom2disease-prediction-ml-model/transformers/default/1/production_symptom_classifier.pkl
✅ Model loaded successfully
   • Diseases: 24
   • Model type: StackingClassifier
   • Test F1: 0.974
   • Test Accuracy: 97.40%
✓ BioBERT loaded on cuda
Loading LLAMA model (this may take 2-3 minutes)...


/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✓ LLAMA loaded on cuda (4-bit)


... (9 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✓ RAG loaded: 44387 documents

✅ All models loaded!

INFO:     103.120.45.57:0 - "OPTIONS /health HTTP/1.1" 200 OK
Grok error: Error code: 400 - {'error': {'message': 'The model `mixtral-8x7b-32768` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
Error: The TF-IDF vectorizer is not fitted
INFO:     103.120.45.58:0 - "POST /chat HTTP/1.1" 500 Internal Server Error
INFO:     103.120.45.57:0 - "OPTIONS /health HTTP/1.1" 200 OK
INFO:     103.120.45.57:0 - "OPTIONS /health HTTP/1.1" 200 OK
INFO:     103.120.45.59:0 - "OPTIONS /health HTTP/1.1" 200 OK
INFO:     103.120.45.59:0 - "OPTIONS /health HTTP/1.1" 200 OK
